# Notebook 05 - Tổng hợp và trực quan hóa

Notebook cuối đọc các kết quả đã lưu, tổng hợp số liệu chính, trực quan hóa cảm xúc theo cụm/n_star và viết insight kinh doanh.

## 0. Khởi tạo môi trường

In [ ]:
from pathlib import Path
import os
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = Path("results/figures")
CSV_DIR = Path("results/csv")
MODEL_DIR = Path("results/models")
for path in [FIG_DIR, CSV_DIR, MODEL_DIR, Path("datasets/cleaned")]:
    path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

## 5.1 Bảng tổng hợp kết quả

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

from src.visualization import plot_sentiment_by_cluster, plot_star_vs_sentiment

train_clean = pd.read_csv("datasets/cleaned/train_clean.csv", encoding="utf-8-sig")
test_clean = pd.read_csv("datasets/cleaned/test_clean.csv", encoding="utf-8-sig")
clustered = pd.read_csv("results/csv/kmeans_clustered.csv", encoding="utf-8-sig")
k_scores = pd.read_csv("results/csv/kmeans_k_scores.csv", encoding="utf-8-sig")
cluster_summary = pd.read_csv("results/csv/kmeans_cluster_summary.csv", encoding="utf-8-sig")
svm_metrics = pd.read_csv("results/csv/svm_metrics_summary.csv", encoding="utf-8-sig").iloc[0]

best_k_row = k_scores.sort_values("silhouette", ascending=False).iloc[0]
best_k = int(best_k_row["k"])
best_sil = best_k_row["silhouette"]
good_clusters = int((cluster_summary["match_ratio"] >= 0.5).sum())

summary_text = f"""
```text
KẾT QUẢ TỔNG HỢP ĐỒ ÁN
Dataset      : {len(train_clean):,} train / {len(test_clean):,} test
Ngôn ngữ     : Tiếng Việt (điện thoại)

PHÂN CỤM (K-Means)
Số cụm K     : {best_k}
Silhouette   : {best_sil:.4f}
Khớp aspect  : {good_clusters}/{len(cluster_summary)} cụm khớp tốt (>=50%)

PHÂN LOẠI (SVM)
Baseline F1  : {svm_metrics['baseline_f1_macro']:.4f}
SVM F1 Macro : {svm_metrics['f1_macro']:.4f}
SVM F1 Weight: {svm_metrics['f1_weighted']:.4f}
CV 5-fold    : {svm_metrics['cv_mean']:.4f} +/- {svm_metrics['cv_std']:.4f}
```
"""
display(Markdown(summary_text))
print("Nhận xét: Bảng trên là phần trả lời định lượng chính cho câu hỏi nghiên cứu về phân cụm và phân loại cảm xúc.")

## 5.2 Cảm xúc theo từng cụm chủ đề

In [ ]:
sentiment_cluster_pct = plot_sentiment_by_cluster(
    clustered, "cluster_name", "sentiment", FIG_DIR / "sentiment_by_cluster.png"
)
display(sentiment_cluster_pct.round(2))

negative_rates = sentiment_cluster_pct["Negative"].sort_values(ascending=False) if "Negative" in sentiment_cluster_pct.columns else pd.Series(dtype=float)
if not negative_rates.empty:
    top_negative_cluster = negative_rates.index[0]
    top_negative_pct = negative_rates.iloc[0]
    print(f"Nhận xét: Chủ đề {top_negative_cluster} có tỷ lệ bình luận tiêu cực cao nhất: {top_negative_pct:.2f}%.")
else:
    print("Nhận xét: Không có cột Negative trong bảng, cần kiểm tra lại nhãn sentiment.")

## 5.3 Cảm xúc theo n_star

In [ ]:
star_sentiment_pct = plot_star_vs_sentiment(train_clean, FIG_DIR / "sentiment_by_star.png")
display(star_sentiment_pct.round(2))
for star in sorted(star_sentiment_pct.index):
    dominant = star_sentiment_pct.loc[star].idxmax()
    pct = star_sentiment_pct.loc[star].max()
    print(f"{star} sao: chủ yếu {dominant} ({pct:.2f}%).")
print("Nhận xét: Nếu xu hướng này rõ, n_star có thể dùng làm tín hiệu tham khảo nhưng không thay thế nhãn aspect-based.")

## 5.4 Insight kinh doanh

In [ ]:
cluster_sentiment = pd.crosstab(clustered["cluster_name"], clustered["sentiment"], normalize="index") * 100
insights = []
if "Negative" in cluster_sentiment.columns:
    worst_topic = cluster_sentiment["Negative"].idxmax()
    worst_pct = cluster_sentiment.loc[worst_topic, "Negative"]
    insights.append(f"Insight 1 - {worst_topic} là điểm cần ưu tiên: {worst_pct:.2f}% bình luận trong chủ đề này là tiêu cực.")
if "Positive" in cluster_sentiment.columns:
    best_topic = cluster_sentiment["Positive"].idxmax()
    best_pct = cluster_sentiment.loc[best_topic, "Positive"]
    insights.append(f"Insight 2 - {best_topic} là điểm mạnh: {best_pct:.2f}% bình luận trong chủ đề này là tích cực.")
if "Neutral" in cluster_sentiment.columns:
    neutral_topic = cluster_sentiment["Neutral"].idxmax()
    neutral_pct = cluster_sentiment.loc[neutral_topic, "Neutral"]
    insights.append(f"Insight 3 - {neutral_topic} có nhiều phản hồi trung lập: {neutral_pct:.2f}%.")

for item in insights:
    display(Markdown(f"- {item}"))

## 5.5 Hạn chế và hướng phát triển

### Hạn chế

- TF-IDF không nắm được ngữ nghĩa sâu và quan hệ phủ định như `không tốt` so với `tốt`.
- K-Means nhạy cảm với khởi tạo ngẫu nhiên và có thể trộn nhiều aspect nếu từ vựng chung quá nhiều.
- Stopwords list còn có thể thiếu từ đặc thù ngành điện thoại.
- Nhãn Neutral khó phân loại vì ranh giới mờ với Positive và Negative.

### Hướng phát triển

- Thay TF-IDF bằng PhoBERT embedding để biểu diễn ngữ nghĩa tiếng Việt tốt hơn.
- Thử LDA, NMF hoặc BERTopic cho topic modeling.
- Phân tích aspect-level sentiment thay vì sentiment tổng thể cho cả bình luận.
- Thu thập thêm dữ liệu hoặc cân bằng lớp để cải thiện F1 Macro.

## 5.6 Kết luận trả lời câu hỏi nghiên cứu

In [ ]:
answer_1 = f"1. K-Means chọn K={best_k}, Silhouette={best_sil:.4f}; có {good_clusters}/{len(cluster_summary)} cụm khớp aspect tốt (>=50%)."
answer_2 = f"2. LinearSVC đạt F1 Macro={svm_metrics['f1_macro']:.4f}, tốt hơn baseline {svm_metrics['baseline_f1_macro']:.4f}."
if not negative_rates.empty:
    answer_3 = f"3. Chủ đề nhận nhiều phản hồi tiêu cực nhất là {top_negative_cluster} với {top_negative_pct:.2f}% Negative."
else:
    answer_3 = "3. Chưa xác định được chủ đề tiêu cực nhất vì thiếu nhãn Negative trong kết quả."

display(Markdown("\n".join([answer_1, answer_2, answer_3])))
print("Đã lưu results/figures/sentiment_by_cluster.png và results/figures/sentiment_by_star.png")